# PnLCalib on clip04 (Kaggle)
Runs PnLCalib on every uploaded frame and saves the camera it finds per frame.

Before running: **Settings → Accelerator → GPU T4 x2**, **Settings → Internet → on**,
and the clip04 frames attached as a dataset (**Add Input**).

Output: `/kaggle/working/pnlcalib_raw_clip04.json` + a few overlay images.
The cameras are in **SoccerNet's frame** (origin = centre spot, 105×68 m pitch, z points down).
Converting to our pitch frame happens locally, when the file is loaded.

In [ ]:
# 1. Get PnLCalib + the single-view weights (main broadcast camera). Kept in /kaggle/temp so
#    the large weight files don't end up in the notebook's output.
!git clone -q https://github.com/mguti97/PnLCalib.git /kaggle/temp/PnLCalib
!pip install -q lsq-ellipse shapely
!wget -q -P /kaggle/temp/PnLCalib/weights https://github.com/mguti97/PnLCalib/releases/download/v1.0.0/SV_kp
!wget -q -P /kaggle/temp/PnLCalib/weights https://github.com/mguti97/PnLCalib/releases/download/v1.0.0/SV_lines
!ls -lh /kaggle/temp/PnLCalib/weights
!nvidia-smi -L

In [ ]:
# 2. Load the two networks (keypoints + line ends).
import glob, json, os, sys, time
os.chdir('/kaggle/temp/PnLCalib'); sys.path.insert(0, '.')

import cv2, numpy as np, torch, yaml
import torchvision.transforms as T
import matplotlib.pyplot as plt

import inference as pnl  # the repo's inference.py; its command-line part doesn't run on import
from model.cls_hrnet import get_cls_net
from model.cls_hrnet_l import get_cls_net as get_cls_net_l
from utils.utils_calib import FramebyFrameCalib

device = 'cuda:0'
pnl.device = device                     # pnl.inference() reads these two as globals
pnl.transform2 = T.Resize((540, 960))   # frames are shrunk to 960x540 before the networks

def load(get_net, cfg, weights):
    net = get_net(yaml.safe_load(open(cfg)))
    net.load_state_dict(torch.load(weights, map_location=device))
    return net.to(device).eval()

model = load(get_cls_net, 'config/hrnetv2_w48.yaml', 'weights/SV_kp')
model_l = load(get_cls_net_l, 'config/hrnetv2_w48_l.yaml', 'weights/SV_lines')
print('models loaded on', device)

In [ ]:
# 3. Calibrate every frame. Same thresholds and PnL refinement as the repo's README command.
frames = sorted(glob.glob('/kaggle/input/**/*.jpg', recursive=True))
assert frames, 'no .jpg found under /kaggle/input: attach the frames dataset (Add Input)'
h, w = cv2.imread(frames[0]).shape[:2]
print(len(frames), 'frames,', w, 'x', h, '| first:', frames[0])

cam = FramebyFrameCalib(iwidth=w, iheight=h, denormalize=True)  # params come out in full-size pixels
results, t0 = [], time.time()
for i, path in enumerate(frames):
    res = pnl.inference(cam, cv2.imread(path), model, model_l, 0.3434, 0.7867, True)
    results.append({'frame': int(os.path.basename(path).split('.')[0]),
                    'ok': res is not None,
                    'rep_err_px': None if res is None else float(res['rep_err']),
                    'cam_params': None if res is None else res['cam_params']})
    if i % 25 == 0:
        print(f'{i:4d}/{len(frames)}  frame {results[-1]["frame"]:5d}  ok={results[-1]["ok"]}  {time.time() - t0:.0f} s')

out = {'clip': 'clip04', 'image_size': [w, h], 'method': 'PnLCalib SV_kp + SV_lines, pnl_refine',
       'world_frame': 'SoccerNet: origin centre spot, 105x68 m, z down', 'frames': results}
json.dump(out, open('/kaggle/working/pnlcalib_raw_clip04.json', 'w'), indent=1, default=float)
errs = [r['rep_err_px'] for r in results if r['ok']]
print(f'done in {time.time() - t0:.0f} s: {len(errs)}/{len(results)} frames calibrated, '
      f'self-reported error median {np.median(errs):.1f} px, max {np.max(errs):.1f} px')

In [ ]:
# 4. Look before trusting: draw PnLCalib's pitch (blue) on a few frames, incl. two of the hard ones.
by_frame = {r['frame']: r for r in results}
fig, axes = plt.subplots(2, 2, figsize=(20, 11.5))
for ax, n in zip(axes.flat, [0, 25, 125, 275]):
    img = cv2.imread(next(p for p in frames if os.path.basename(p) == f'{n:05d}.jpg'))
    r = by_frame[n]
    if r['ok']:
        img = pnl.project(img, pnl.projection_from_cam_params(r))
    cv2.imwrite(f'/kaggle/working/overlay_{n:05d}.jpg', img)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax.set_axis_off()
    ax.set_title(f'frame {n}: ' + (f"self-reported {r['rep_err_px']:.1f} px" if r['ok'] else 'FAILED'))
plt.tight_layout(); plt.show()